# House Prices (Kaggle) — modelo da competição e comparação de otimizadores

> Alan Hoffmann Dos Santos

Parte 1: limpeza, engenharia de atributos e modelos (árvores, lineares e ensemble) com validação cruzada 5-fold em RMSE do log.
Parte 2: comparação de otimizadores em uma rede neural (MLP) usando o dataset MPG.

Antes de rodar: envie `train.csv` e `test.csv` para `/dados/` e `mpg_limpo2.csv` também para `/dados/`.

In [2]:
!pip install -q xgboost
import os, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import nnls
from scipy.stats import skew
from scipy.special import boxcox1p
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import Lasso, ElasticNet, RidgeCV
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.base import clone
import xgboost as xgb

SEED = 42
random.seed(SEED); np.random.seed(SEED)


## 1. Carregamento e limpeza
Remove as 2 casas com mais de 4.000 pés² vendidas abaixo de US$ 300 mil. Estatísticas de preenchimento vêm só do treino.

In [ ]:
DADOS = 'C:/Users/Alan/Desktop/HousePrices/HousePrices/Trabalho_HousePrices/dados/'
train = pd.read_csv(DADOS + 'train.csv', keep_default_na=False, na_values=['NA', ''])
test = pd.read_csv(DADOS + 'test.csv', keep_default_na=False, na_values=['NA', ''])

outliers = (train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000)
train = train.loc[~outliers].reset_index(drop=True)
print('Treino após remover outliers:', train.shape, '| teste:', test.shape)

n_train = len(train)
test_id = test['Id']
y = np.log1p(train['SalePrice'].values)
dados = pd.concat([train.drop(columns='SalePrice'), test]).reset_index(drop=True).drop(columns='Id')
ref = dados.iloc[:n_train]

# Faltantes que significam "não possui"
nao_possui = ['PoolQC','MiscFeature','Alley','Fence','FireplaceQu','GarageType','GarageFinish','GarageQual',
              'GarageCond','BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','MasVnrType']
for c in nao_possui:
    dados[c] = dados[c].fillna('None')
dados['Electrical'] = dados['Electrical'].fillna(ref['Electrical'].mode()[0])
for c in ['MSZoning','Functional','Utilities','Exterior1st','Exterior2nd','SaleType','KitchenQual']:
    dados[c] = dados[c].fillna(ref[c].mode()[0])
for c in dados.select_dtypes('number').columns:
    dados[c] = dados[c].fillna(ref[c].median())
for c in ['GarageYrBlt','MasVnrArea','LotFrontage']:
    dados[c] = dados[c].fillna(ref[c].median())
print('Faltantes restantes:', int(dados.isna().sum().sum()))


FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/Alan/Desktop/HousePrices/Trabalho_HousePrices/dados/train.csv'

## 2. Engenharia de atributos
Códigos viram categorias, qualidades viram 0–5 e algumas áreas e idades são combinadas.

In [ ]:
for c in ['MSSubClass','MoSold','YrSold']:
    dados[c] = dados[c].astype(str)

qual = {'None':0,'Po':1,'Fa':2,'TA':3,'Gd':4,'Ex':5}
for c in ['ExterQual','ExterCond','BsmtQual','BsmtCond','HeatingQC','KitchenQual','FireplaceQu','GarageQual','GarageCond']:
    dados[c] = dados[c].map(qual).astype(int)

dados['TotalSF'] = dados['TotalBsmtSF'] + dados['1stFlrSF'] + dados['2ndFlrSF']
dados['QualTotSF'] = dados['OverallQual'] * dados['TotalSF']
dados['TotalBath'] = dados['FullBath'] + 0.5*dados['HalfBath'] + dados['BsmtFullBath'] + 0.5*dados['BsmtHalfBath']
dados['HouseAge'] = dados['YrSold'].astype(int) - dados['YearBuilt']
dados['RemodAge'] = dados['YrSold'].astype(int) - dados['YearRemodAdd']
dados['HasPool'] = (dados['PoolArea'] > 0).astype(int)


## 3. Matrizes para modelos
Árvores recebem só one-hot. Modelos lineares e a rede recebem também Box-Cox nas variáveis assimétricas.

In [ ]:
X_arv = pd.get_dummies(dados).astype(float)
lin = dados.copy()
num = lin.select_dtypes('number').columns
sk = lin.iloc[:n_train][num].apply(lambda s: skew(s)).abs()
for c in sk[sk > 0.5].index:
    lin[c] = boxcox1p(lin[c].clip(lower=0), 0.15)
X_lin = pd.get_dummies(lin).astype(float)

X_arv_tr, X_arv_te = X_arv.iloc[:n_train].values, X_arv.iloc[n_train:].values
X_lin_tr, X_lin_te = X_lin.iloc[:n_train].values, X_lin.iloc[n_train:].values
print('Matriz árvores:', X_arv_tr.shape, '| matriz lineares:', X_lin_tr.shape)


## 4. Validação cruzada
5 partes, RMSE do log, a mesma métrica do Kaggle.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

modelos = {
    'Árvore (GridSearch)': None,
    'Ridge': (make_pipeline(RobustScaler(), RidgeCV(alphas=np.logspace(-1, 2.5, 40))), 'lin'),
    'Lasso': (make_pipeline(RobustScaler(), Lasso(alpha=0.0005, max_iter=50000)), 'lin'),
    'ElasticNet': (make_pipeline(RobustScaler(), ElasticNet(alpha=0.0006, l1_ratio=0.8, max_iter=50000)), 'lin'),
    'Random Forest': (RandomForestRegressor(n_estimators=500, max_features=0.3, n_jobs=-1, random_state=SEED), 'arv'),
    'Extra Trees': (ExtraTreesRegressor(n_estimators=500, max_features=0.3, n_jobs=-1, random_state=SEED), 'arv'),
    'Gradient Boosting': (GradientBoostingRegressor(n_estimators=2000, learning_rate=0.02, max_depth=4,
                          max_features='sqrt', subsample=0.8, random_state=SEED), 'arv'),
    'XGBoost': (xgb.XGBRegressor(n_estimators=2000, learning_rate=0.02, max_depth=3, subsample=0.8,
                colsample_bytree=0.6, random_state=SEED, n_jobs=-1), 'arv'),
}

grade = {'max_depth': [3,4,5,6,8,10,None], 'min_samples_leaf': [1,3,5,10,20]}
busca = GridSearchCV(DecisionTreeRegressor(random_state=SEED), grade,
                     scoring='neg_root_mean_squared_error', cv=kf, n_jobs=-1).fit(X_arv_tr, y)
print('Árvore - melhores parâmetros:', busca.best_params_, '| RMSE:', round(-busca.best_score_, 4))
modelos['Árvore (GridSearch)'] = (busca.best_estimator_, 'arv')

oof, notas = {}, {}
for nome, (m, tipo) in modelos.items():
    Xtr = X_lin_tr if tipo == 'lin' else X_arv_tr
    t0 = time.time()
    oof[nome] = cross_val_predict(clone(m), Xtr, y, cv=kf)
    notas[nome] = rmse(oof[nome], y)
    print(f'{nome:22s} RMSE(log) = {notas[nome]:.4f}  ({time.time()-t0:.0f}s)')


## 5. Ensemble com pesos NNLS
Os pesos saem das previsões out-of-fold, sem olhar o teste.

In [ ]:
OOF = pd.DataFrame(oof)
w, _ = nnls(OOF.values, y)
w = w / w.sum()
pesos = pd.Series(w, index=OOF.columns).round(3)
print(pesos[pesos > 0].sort_values(ascending=False))
print('RMSE do ensemble (in-sample dos pesos):', round(rmse(OOF.values @ w, y), 4))


## 6. Treino final e submissão
Treina cada modelo com todos os dados de treino e aplica os pesos no teste.

In [ ]:
preds = {}
for nome, (m, tipo) in modelos.items():
    if pesos[nome] <= 0:
        continue
    Xtr = X_lin_tr if tipo == 'lin' else X_arv_tr
    Xte = X_lin_te if tipo == 'lin' else X_arv_te
    preds[nome] = clone(m).fit(Xtr, y).predict(Xte)

final_log = sum(pesos[n] * preds[n] for n in preds) / sum(pesos[n] for n in preds)
submissao = pd.DataFrame({'Id': test_id, 'SalePrice': np.expm1(final_log)})
submissao.to_csv('/content/submission.csv', index=False)
print(submissao.shape, '| vazios:', int(submissao.isna().sum().sum()), '| colunas:', list(submissao.columns))
submissao.head()


## 7. Comparação de otimizadores (MLP no dataset MPG)
Mesma arquitetura, mesma semente, mesma divisão e early stopping. Só o otimizador muda.

In [ ]:
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

mpg = pd.read_csv('/content/mpg_limpo2.csv')
alvo = 'km_l'
entradas = [c for c in mpg.columns if c not in (alvo, 'name', 'origin')]
X = pd.get_dummies(mpg[entradas + ['origin']], columns=['origin']).astype(float).values
y_mpg = mpg[alvo].values.astype(float)

X_tr, X_te, y_tr, y_te = train_test_split(X, y_mpg, test_size=0.2, random_state=SEED)
esc = StandardScaler().fit(X_tr)
X_tr, X_te = esc.transform(X_tr).astype('float32'), esc.transform(X_te).astype('float32')

def criar_rede(seed=SEED):
    tf.random.set_seed(seed); np.random.seed(seed)
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(X_tr.shape[1],)),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(1),
    ])

otimizadores = {
    'SGD': lambda: tf.keras.optimizers.SGD(learning_rate=0.01),
    'SGD+Momentum': lambda: tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    'RMSprop': lambda: tf.keras.optimizers.RMSprop(learning_rate=0.001),
    'Adagrad': lambda: tf.keras.optimizers.Adagrad(learning_rate=0.01),
    'Adadelta': lambda: tf.keras.optimizers.Adadelta(learning_rate=1.0),
    'Adam': lambda: tf.keras.optimizers.Adam(learning_rate=0.001),
    'AdamW': lambda: tf.keras.optimizers.AdamW(learning_rate=0.001),
    'Nadam': lambda: tf.keras.optimizers.Nadam(learning_rate=0.001),
}

resultados, curvas = [], {}
for nome, fabrica in otimizadores.items():
    net = criar_rede()
    net.compile(optimizer=fabrica(), loss='mse')
    t0 = time.time()
    hist = net.fit(X_tr, y_tr, validation_split=0.2, epochs=300, batch_size=32, verbose=0,
                   callbacks=[tf.keras.callbacks.EarlyStopping(patience=30, restore_best_weights=True)])
    pred = net.predict(X_te, verbose=0).ravel()
    curvas[nome] = hist.history
    resultados.append({'otimizador': nome, 'RMSE_teste': np.sqrt(mean_squared_error(y_te, pred)),
                       'MAE_teste': mean_absolute_error(y_te, pred), 'R2_teste': r2_score(y_te, pred),
                       'melhor_epoca': int(np.argmin(hist.history['val_loss']) + 1),
                       'tempo_s': round(time.time() - t0, 1)})

res = pd.DataFrame(resultados).sort_values('RMSE_teste')
res.to_csv('/content/comparacao_otimizadores_mpg.csv', index=False)
res.round(3)


## 8. Gráfico das curvas de perda

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for nome, h in curvas.items():
    ax.plot(h['val_loss'], label=nome)
ax.set_yscale('log'); ax.set_xlabel('Época'); ax.set_ylabel('MSE validação (log)')
ax.set_title('Perda de validação por otimizador (MPG)'); ax.legend(fontsize=8); ax.grid(alpha=0.2)
plt.show()
